In [ ]:
pip install openjij

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 12.9 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [ ]:
import pandas as pd
import numpy as np
import openjij as oj
import os

In [ ]:
# === 1. ファイル読み込み ===
# ファイル設定
file_path = "山陽IC_INPUT.xlsx"

if not os.path.exists(file_path):
    print(f"エラー: '{file_path}' が見つかりません。")
else:
    df = pd.read_excel(file_path)

    # B列: 荷物ナンバー / C列: 配送先
    item_ids_raw = df.iloc[:, 1].dropna().astype(str).tolist()
    destinations_raw = df.iloc[:, 2].dropna().astype(str).tolist()

    # ヘッダー行や数値以外のデータを除外してクレンジング
    valid_indices = [idx for idx, val in enumerate(item_ids_raw) if val.isdigit()]
    item_ids = [item_ids_raw[i] for i in valid_indices]
    destinations = [destinations_raw[i] for i in valid_indices]

    n_items = len(destinations)
    print(f"✅ Section 1 完了: 荷物 {n_items} 件を正常に読み込みました。")

✅ Section 1 完了: 荷物 93 件を正常に読み込みました。


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [ ]:
# === Section 2: 最適化モデル（QUBO）の構築 ===

# 設定の微調整
n_trucks = 20
n_slots = 7
n_variables = n_items * n_trucks * n_slots

# 重みのバランスを極端に変更
lam_unique = 50000  # 重複禁止（最優先：前回の2.5倍）
lam_cluster = 800   # 行き先集約（重複よりは弱く、しかし確実に効く程度）
lam_truck_cost = 200

def get_idx(i, v, p):
    return i * (n_trucks * n_slots) + v * n_slots + p

QUBO = {}
def add_qubo(i, j, val):
    if i > j: i, j = j, i
    QUBO[(i, j)] = QUBO.get((i, j), 0) + val

# (A) 一意制約：Σ_{v,p} x_{i,v,p} = 1
# これを (Σx - 1)^2 = Σx^2 + 2Σxx - 2Σx と展開
for i in range(n_items):
    indices = [get_idx(i, v, p) for v in range(n_trucks) for p in range(n_slots)]
    for idx1 in indices:
        add_qubo(idx1, idx1, -1 * lam_unique) # -2Σx の項（係数調整済み）
        for idx2 in indices:
            if idx1 >= idx2: continue
            add_qubo(idx1, idx2, 2 * lam_unique) # 2Σxx の項

# (B) スロット排他制約：同じトラックの同じスロットには1つしか荷物を置けない
# これにより1スロットに複数荷物が重なり、上限を突破するのを防ぐ
for v in range(n_trucks):
    for p in range(n_slots):
        indices_slot = [get_idx(i, v, p) for i in range(n_items)]
        for idx1 in indices_slot:
            for idx2 in indices_slot:
                if idx1 >= idx2: continue
                add_qubo(idx1, idx2, lam_unique * 1.5) # 強力な衝突禁止

# (C) 同一目的地集約
for v in range(n_trucks):
    indices_v = [get_idx(i, v, p) for i in range(n_items) for p in range(n_slots)]
    for idx1 in indices_v:
        for idx2 in indices_v:
            if idx1 >= idx2: continue
            i1, i2 = idx1 // (n_trucks * n_slots), idx2 // (n_trucks * n_slots)
            # 同じ目的地なら報酬
            if destinations[i1] == destinations[i2]:
                add_qubo(idx1, idx2, -lam_cluster)

print(f"✅ QUBO再構築完了: 重複禁止を強化し、スロット排他制約を追加しました。")

✅ QUBO再構築完了: 重複禁止を強化し、スロット排他制約を追加しました。


In [ ]:
 # === 3. 実行 ===
print(" 最適化計算を実行中...")
sampler = oj.SASampler()
result = sampler.sample_qubo(QUBO, num_reads=150) # 回数を増やして精度重視
best_solution = result.record[0][0]

print("✅ Section 3 完了: 計算が終了しました。")

 最適化計算を実行中...
✅ Section 3 完了: 計算が終了しました。


In [ ]:
# === 4. 結果表示 ===
print(f"--- 運行計画（行き先集約・No.順ソート） ---\n")
used_trucks = 0
total_loaded = 0

for v in range(n_trucks):
    truck_items = []
    for p in range(n_slots):
        for i in range(n_items):
            if best_solution[get_idx(i, v, p)] == 1:
                truck_items.append((int(item_ids[i]), destinations[i]))
                total_loaded += 1

    if truck_items:
        used_trucks += 1
        # 荷物No.で昇順ソート
        truck_items.sort(key=lambda x: x[0])

        # 整形して表示
        display_list = [f"No.{item[0]}({item[1]})" for item in truck_items]
        print(f"トラック {used_trucks:02d} [{len(truck_items)}台]: " + " | ".join(display_list))

print(f"\n✅ Section 4 完了: 全{n_items}件中、{total_loaded}件を {used_trucks}台に集約しました。")

--- 運行計画（行き先集約・No.順ソート） ---

トラック 01 [7台]: No.1(真庭B → 中古車C) | No.2(真庭B → 中古車C) | No.3(中古車C → 高野店) | No.4(中古車C → 高野店) | No.5(高野店) | No.6(高野店 → 中古車C) | No.7(高野店 → 中古車C)
トラック 02 [7台]: No.8(高野店 → 中古車C) | No.9(高野店 → 中古車C) | No.10(高野店 → 赤磐C) | No.11(津山B) | No.12(津山B) | No.13(津山B) | No.14(津山B → 中古車C)
トラック 03 [7台]: No.15(津山B → 中古車C) | No.16(津山B → 中古車C) | No.17(津山店 → 中古車C) | No.18(津山店 → 中古車C) | No.19(水島店) | No.20(中古車C → 吉備路店) | No.21(吉備路店 → 中古車C)
トラック 04 [7台]: No.22(中古車C → 吉備路店) | No.23(吉備路店) | No.24(吉備路店 → 中古車C) | No.25(吉備路店 → 中古車C) | No.26(倉敷中央店) | No.27(倉敷中島店) | No.28(中古車C → 倉敷中島店)
トラック 05 [7台]: No.29(倉敷中島店) | No.30(中庄店 → 中古車C) | No.31(平島店) | No.32(平島店 → 中古車C) | No.33(平島店 → 中古車C) | No.34(平島店 → 中古車C) | No.35(十日市店（岡山B）)
トラック 06 [7台]: No.36(十日市店（岡山B）) | No.37(高屋店 → 中古車C) | No.38(高屋店 → 中古車C) | No.39(高屋店 → 中古車C) | No.40(中古車C → 十日市店) | No.41(十日市店) | No.42(十日市店)
トラック 07 [7台]: No.43(十日市店 → 中古車C) | No.44(十日市店 → 中古車C) | No.45(十日市店 → 中古車C) | No.46(十日市店 → 中古車C) | No.47(中古車C → 玉野紅陽台店) | No.48(中古車C → 玉野紅陽

In [ ]:
# === Section 2 & 3: 制約絶対遵守型モデル ===
import random

n_trucks = 20
n_slots = 7
# 罰金をさらに引き上げ
lam_unique = 100000
# ボーナスを「罰金の1/5以下」に抑える（重要：制約を上回らせない）
lam_cluster = 15000
lam_truck_cost = 0

# ランダム性を加える
calc_indices = list(range(n_items))
random.shuffle(calc_indices)

QUBO = {}
def add_qubo(i, j, val):
    if i > j: i, j = j, i
    QUBO[(i, j)] = QUBO.get((i, j), 0) + val

# (A) 一意制約：各荷物は1回だけ（係数を2倍に強化）
for i in range(n_items):
    indices = [get_idx(i, v, p) for v in range(n_trucks) for p in range(n_slots)]
    for idx1 in indices:
        add_qubo(idx1, idx1, -lam_unique)
        for idx2 in indices:
            if idx1 >= idx2: continue
            add_qubo(idx1, idx2, 2 * lam_unique)

# (B) スロット排他制約：これが「台数制限」の正体
# 「1つのトラックの1つのスロット」には絶対に1つしか載せない
for v in range(n_trucks):
    for p in range(n_slots):
        indices_slot = [get_idx(i, v, p) for i in range(n_items)]
        for idx1 in indices_slot:
            for idx2 in indices_slot:
                if idx1 >= idx2: continue
                # 重複禁止よりも重い罰金を課す
                add_qubo(idx1, idx2, lam_unique * 3)

# (C) 同一目的地集約：報酬は制約を壊さない範囲で
for v in range(n_trucks):
    indices_v = [get_idx(i, v, p) for i in range(n_items) for p in range(n_slots)]
    for idx1 in indices_v:
        for idx2 in indices_v:
            if idx1 >= idx2: continue
            i1, i2 = idx1 // (n_trucks * n_slots), idx2 // (n_trucks * n_slots)
            if destinations[i1] == destinations[i2]:
                add_qubo(idx1, idx2, -lam_cluster)

# 実行（試行回数を増やして安定させる）
sampler = oj.SASampler()
result = sampler.sample_qubo(QUBO, num_reads=150)
best_solution = result.record[0][0]

In [ ]:
 # === 3. 実行 ===
print(" 最適化計算を実行中...")
# 実行（試行回数を増やして安定させる）
sampler = oj.SASampler()
result = sampler.sample_qubo(QUBO, num_reads=150)
best_solution = result.record[0][0]

print("✅ Section 3 完了: 計算が終了しました。")

 最適化計算を実行中...
✅ Section 3 完了: 計算が終了しました。


In [ ]:
# === 4. 結果表示 ===
print(f"--- 運行計画（行き先集約・No.順ソート） ---\n")
used_trucks = 0
total_loaded = 0

for v in range(n_trucks):
    truck_items = []
    for p in range(n_slots):
        for i in range(n_items):
            if best_solution[get_idx(i, v, p)] == 1:
                truck_items.append((int(item_ids[i]), destinations[i]))
                total_loaded += 1

    if truck_items:
        used_trucks += 1
        # 荷物No.で昇順ソート
        truck_items.sort(key=lambda x: x[0])

        # 整形して表示
        display_list = [f"No.{item[0]}({item[1]})" for item in truck_items]
        print(f"トラック {used_trucks:02d} [{len(truck_items)}台]: " + " | ".join(display_list))

print(f"\n✅ Section 4 完了: 全{n_items}件中、{total_loaded}件を {used_trucks}台に集約しました。")

--- 運行計画（行き先集約・No.順ソート） ---

トラック 01 [7台]: No.1(真庭B → 中古車C) | No.2(真庭B → 中古車C) | No.3(中古車C → 高野店) | No.4(中古車C → 高野店) | No.5(高野店) | No.6(高野店 → 中古車C) | No.7(高野店 → 中古車C)
トラック 02 [7台]: No.8(高野店 → 中古車C) | No.9(高野店 → 中古車C) | No.10(高野店 → 赤磐C) | No.11(津山B) | No.12(津山B) | No.13(津山B) | No.14(津山B → 中古車C)
トラック 03 [7台]: No.15(津山B → 中古車C) | No.16(津山B → 中古車C) | No.17(津山店 → 中古車C) | No.18(津山店 → 中古車C) | No.19(水島店) | No.20(中古車C → 吉備路店) | No.21(吉備路店 → 中古車C)
トラック 04 [7台]: No.22(中古車C → 吉備路店) | No.23(吉備路店) | No.24(吉備路店 → 中古車C) | No.25(吉備路店 → 中古車C) | No.26(倉敷中央店) | No.27(倉敷中島店) | No.28(中古車C → 倉敷中島店)
トラック 05 [7台]: No.29(倉敷中島店) | No.30(中庄店 → 中古車C) | No.31(平島店) | No.32(平島店 → 中古車C) | No.33(平島店 → 中古車C) | No.34(平島店 → 中古車C) | No.35(十日市店（岡山B）)
トラック 06 [7台]: No.36(十日市店（岡山B）) | No.37(高屋店 → 中古車C) | No.38(高屋店 → 中古車C) | No.39(高屋店 → 中古車C) | No.40(中古車C → 十日市店) | No.41(十日市店) | No.42(十日市店)
トラック 07 [7台]: No.43(十日市店 → 中古車C) | No.44(十日市店 → 中古車C) | No.45(十日市店 → 中古車C) | No.46(十日市店 → 中古車C) | No.47(中古車C → 玉野紅陽台店) | No.48(中古車C → 玉野紅陽

In [ ]:
# === Section 2 & 3: 行き先集約・トラック分散許容モデル ===
import random

n_trucks = 20
n_slots = 7
lam_unique = 100000
lam_cluster = 30000  # 集約報酬を大幅に強化
lam_truck_cost = 0   # 台数削減を考えず、まとめやすさを優先

# 荷物の順序を完全にランダム化
calc_indices = list(range(n_items))
random.shuffle(calc_indices)

QUBO = {}
def add_qubo(i, j, val):
    if i > j: i, j = j, i
    QUBO[(i, j)] = QUBO.get((i, j), 0) + val

# (A) 一意制約
for i in range(n_items):
    indices = [get_idx(i, v, p) for v in range(n_trucks) for p in range(n_slots)]
    for idx1 in indices:
        add_qubo(idx1, idx1, -lam_unique)
        for idx2 in indices:
            if idx1 >= idx2: continue
            add_qubo(idx1, idx2, 2 * lam_unique)

# (B) スロット排他制約（1枠1荷物を絶対守る）
for v in range(n_trucks):
    for p in range(n_slots):
        indices_slot = [get_idx(i, v, p) for i in range(n_items)]
        for idx1 in indices_slot:
            for idx2 in indices_slot:
                if idx1 >= idx2: continue
                add_qubo(idx1, idx2, lam_unique * 2.0)

# (C) 同一目的地集約：磁力を最大化
for v in range(n_trucks):
    indices_v = [get_idx(i, v, p) for i in range(n_items) for p in range(n_slots)]
    for idx1 in indices_v:
        for idx2 in indices_v:
            if idx1 >= idx2: continue
            i1, i2 = idx1 // (n_trucks * n_slots), idx2 // (n_trucks * n_slots)
            if destinations[i1] == destinations[i2]:
                # 非常に強い「引き寄せ」を設定
                add_qubo(idx1, idx2, -lam_cluster)

In [ ]:
# 実行（num_readsを増やして「遠くの解」を見つけやすくする）
sampler = oj.SASampler()
result = sampler.sample_qubo(QUBO, num_reads=300)
best_solution = result.record[0][0]

In [ ]:
# === 4. 結果表示 ===
print(f"--- 運行計画（行き先集約・No.順ソート） ---\n")
used_trucks = 0
total_loaded = 0

for v in range(n_trucks):
    truck_items = []
    for p in range(n_slots):
        for i in range(n_items):
            if best_solution[get_idx(i, v, p)] == 1:
                truck_items.append((int(item_ids[i]), destinations[i]))
                total_loaded += 1

    if truck_items:
        used_trucks += 1
        # 荷物No.で昇順ソート
        truck_items.sort(key=lambda x: x[0])

        # 整形して表示
        display_list = [f"No.{item[0]}({item[1]})" for item in truck_items]
        print(f"トラック {used_trucks:02d} [{len(truck_items)}台]: " + " | ".join(display_list))

print(f"\n✅ Section 4 完了: 全{n_items}件中、{total_loaded}件を {used_trucks}台に集約しました。")

--- 運行計画（行き先集約・No.順ソート） ---

トラック 01 [7台]: No.1(真庭B → 中古車C) | No.2(真庭B → 中古車C) | No.3(中古車C → 高野店) | No.4(中古車C → 高野店) | No.5(高野店) | No.6(高野店 → 中古車C) | No.7(高野店 → 中古車C)
トラック 02 [7台]: No.8(高野店 → 中古車C) | No.9(高野店 → 中古車C) | No.10(高野店 → 赤磐C) | No.11(津山B) | No.12(津山B) | No.13(津山B) | No.14(津山B → 中古車C)
トラック 03 [7台]: No.15(津山B → 中古車C) | No.16(津山B → 中古車C) | No.17(津山店 → 中古車C) | No.18(津山店 → 中古車C) | No.19(水島店) | No.20(中古車C → 吉備路店) | No.21(吉備路店 → 中古車C)
トラック 04 [7台]: No.22(中古車C → 吉備路店) | No.23(吉備路店) | No.24(吉備路店 → 中古車C) | No.25(吉備路店 → 中古車C) | No.26(倉敷中央店) | No.27(倉敷中島店) | No.28(中古車C → 倉敷中島店)
トラック 05 [7台]: No.29(倉敷中島店) | No.30(中庄店 → 中古車C) | No.31(平島店) | No.32(平島店 → 中古車C) | No.33(平島店 → 中古車C) | No.34(平島店 → 中古車C) | No.35(十日市店（岡山B）)
トラック 06 [7台]: No.36(十日市店（岡山B）) | No.37(高屋店 → 中古車C) | No.38(高屋店 → 中古車C) | No.39(高屋店 → 中古車C) | No.40(中古車C → 十日市店) | No.41(十日市店) | No.42(十日市店)
トラック 07 [7台]: No.43(十日市店 → 中古車C) | No.44(十日市店 → 中古車C) | No.45(十日市店 → 中古車C) | No.46(十日市店 → 中古車C) | No.46(十日市店 → 中古車C) | No.47(中古車C → 玉野紅陽台店